# L7c: Restricted Boltzmann Machines
In this lecture, we'll continue our discussion of Boltzmann Machines by introducing the __Restricted Boltzmann Machine (RBM)__. 

A Restricted Boltzmann Machine (RBM) is a simplified version of the general Boltzmann machine that removes connections between nodes in the same layer. This restriction creates a bipartite graph structure that enables efficient training and sampling algorithms.

> __Learning Objectives__
>
> * __Define a restricted Boltzmann machine:__ Describe the bipartite graph structure of an RBM with visible and hidden layers, and explain how the absence of intra-layer connections simplifies the network.
> * __Implement feedforward and feedback passes:__ Apply the conditional independence property to compute hidden states from visible inputs and reconstruct visible states from hidden activations.
> * __Explain contrastive divergence training:__ Describe the CD-k algorithm for approximating the gradient of the log-likelihood and updating the weights and biases of the RBM.

Let's get started!

___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Understanding a Small Boltzmann Machines](CHEME-5820-L7a-Example-SmallBoltzmannMachine-Spring-2026.ipynb). In this example, we will explore some questions surrounding the sampling and stationary distribution of a _small_ Boltzmann machine. In particular, we'll look at one of the key limitations of any training approach, namely requiring convergence to a stationary distribution for each training iteration.
___

<div>
    <center>
      <img
        src="figs/Fig-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="200"
        width="600"
      />
    </center>
  </div>

## Review: Boltzmann Machines
Formally, [a Boltzmann Machine](https://en.wikipedia.org/wiki/Boltzmann_machine) $\mathcal{B}$ is a fully connected _undirected weighted graph_ defined by the tuple $\mathcal{B} = \left(\mathcal{V},\mathcal{E}, \mathbf{W},\mathbf{b}, \mathbf{s}\right)$.
* __Units__: Each unit (vertex, node, neuron) $v_{i}\in\mathcal{V}$ has a binary state (`on` or `off`) and a bias value 
$b_{i}\in\mathbb{R}$. The bias vector $\mathbf{b}\in\mathbb{R}^{|\mathcal{V}|}$ is the vector of bias values for all nodes in the network.
    - A machine $\mathcal{B}$ may have _visible_ nodes (the state is visible) and _hidden_ nodes (the state is not visible to us). The visible nodes represent the state of the data, while the hidden nodes capture the underlying structure of the data (latent variables).
    - The set of all nodes in the machine $\mathcal{B}$ is denoted by $\mathcal{V} \equiv \left\{v_{1},v_{2},\ldots,v_{|\mathcal{V}|}\right\}$, where $|\mathcal{V}|$ is the number of nodes in the network. We can partition the set of nodes into visible nodes $\mathcal{V}_{\text{vis}}$ and hidden nodes $\mathcal{V}_{\text{hid}}$, such that $\mathcal{V} = \mathcal{V}_{\text{vis}} \cup \mathcal{V}_{\text{hid}}$ and $\mathcal{V}_{\text{vis}} \cap \mathcal{V}_{\text{hid}} = \emptyset$.
* __Edges__: Each edge $e\in\mathcal{E}$ has a weight. The weight of the edge connecting $v_{i}\in\mathcal{V}$ and $v_{j}\in\mathcal{V}$, is denoted by $w_{ij}\in\mathbf{W}$, where the weight matrix $\mathbf{W}\in\mathbb{R}^{|\mathcal{V}|\times|\mathcal{V}|}$ is symmetric, i.e. $w_{ij} = w_{ji}$ and $w_{ii} = 0$ (no self loops). The weights $w_{ij}\in\mathbb{R}$ determine the strength of the connection between nodes $i$ and $j$. 
* __States__: The state of the machine $\mathcal{B}$ is represented by a binary vector $\mathbf{s}\in\mathbb{R}^{|\mathcal{V}|}$, where $s_{i}\in\{-1,1\}$ is the state of node $v_{i}$. When $s_{i} = 1$, the node is `on`, and when $s_{i} = -1$, the node is `off`. The set of all possible _state configurations_ is denoted by $\mathcal{S} \equiv \left\{\mathbf{s}^{(1)},\mathbf{s}^{(2)},\ldots,\mathbf{s}^{(N)}\right\}$, where $N$ is the number of possible state configurations, or $N = 2^{|\mathcal{V}|}$ for binary units.

Let's take another look at the Lab L7b from a different perspective. 

> [▶ Understanding a Small Boltzmann Machines](CHEME-5820-L7a-Example-SmallBoltzmannMachine-Spring-2026.ipynb). In this example, we will explore some questions surrounding the sampling and stationary distribution of a _small_ Boltzmann machine. In particular, we'll look at one of the key limitations of any training approach, namely requiring convergence to a stationary distribution for each training iteration.

Now that we have formally defined the machine $\mathcal{B}$, let's take a look at a restricted version of this machine, the Restricted Boltzmann Machine (RBM).

___

<div>
    <center>
      <img
        src="figs/Fig-Restricted-Boltzmann-Machine-Schematic.svg"
        alt="triangle with all three sides equal"
        height="200"
        width="600"
      />
    </center>
</div>

## Restricted Boltzmann Machines (RBMs)
A general Boltzmann machine is a fully connected undirected graph where every node connects to every other node. A Restricted Boltzmann Machine simplifies this structure by organizing nodes into two layers: a visible layer and a hidden layer, with connections only between layers.

Formally, an RBM is defined by the tuple $\mathcal{R} = \left(\mathcal{V}, \mathcal{H}, \mathbf{W}, \mathbf{a}, \mathbf{b}\right)$:

* **Visible layer** $\mathcal{V}$: The set of visible units $\left\{v_{1}, v_{2}, \ldots, v_{n}\right\}$ where $n = |\mathcal{V}|$. Each visible unit has a binary state $v_{i} \in \{-1, 1\}$ and a bias $a_{i} \in \mathbb{R}$. The bias vector is $\mathbf{a} \in \mathbb{R}^{n}$.
* **Hidden layer** $\mathcal{H}$: The set of hidden units $\left\{h_{1}, h_{2}, \ldots, h_{m}\right\}$ where $m = |\mathcal{H}|$. Each hidden unit has a binary state $h_{j} \in \{-1, 1\}$ and a bias $b_{j} \in \mathbb{R}$. The bias vector is $\mathbf{b} \in \mathbb{R}^{m}$.
* **Weight matrix** $\mathbf{W} \in \mathbb{R}^{n \times m}$: The weight $w_{ij}$ represents the connection strength between visible unit $v_{i}$ and hidden unit $h_{j}$.

The restriction is that there are no connections within the visible layer (no $v_{i}$-$v_{j}$ edges) and no connections within the hidden layer (no $h_{i}$-$h_{j}$ edges). This bipartite structure is the source of the name "restricted."

### Sampling and CD-$k$ for RBMs (Adapted from General Boltzmann Machines)
In a general Boltzmann machine, Gibbs sampling updates one node at a time because all nodes can be coupled. In an RBM, the bipartite structure removes intra-layer dependencies, so we can replace single-node Gibbs updates with **block Gibbs updates**:

1. sample all hidden units in parallel given visible units;
2. sample all visible units in parallel given hidden units.

For spin states in $\{-1,+1\}$, the conditional probabilities are:
$$
P(h_{j}=1\mid\mathbf{v}) = \frac{1}{1+\exp\left(-2\beta\left((\mathbf{W}^{\top}\mathbf{v})_{j}+b_{j}\right)\right)}
$$
and
$$
P(v_{i}=1\mid\mathbf{h}) = \frac{1}{1+\exp\left(-2\beta\left((\mathbf{W}\mathbf{h})_{i}+a_{i}\right)\right)}.
$$

This gives the RBM version of CD-$k$:

1. **Positive phase (data):** clamp $\mathbf{v}^{(0)}=\mathbf{x}^{(r)}$, compute $\mathbf{p}^{+}=P(\mathbf{h}=1\mid\mathbf{v}^{(0)})$, and sample $\mathbf{h}^{(0)}$.
2. **Negative phase (model):** run $k$ block Gibbs steps
   $\mathbf{v}^{(t)}\sim P(\mathbf{v}\mid\mathbf{h}^{(t-1)})$,
   $\mathbf{h}^{(t)}\sim P(\mathbf{h}\mid\mathbf{v}^{(t)})$ for $t=1,\dots,k$.
3. **Parameter update:** using $\mathbf{p}^{-}=P(\mathbf{h}=1\mid\mathbf{v}^{(k)})$,
$$
\Delta\mathbf{W}=\eta\left(\mathbf{v}^{(0)}(\mathbf{p}^{+})^{\top}-\mathbf{v}^{(k)}(\mathbf{p}^{-})^{\top}\right),
\quad
\Delta\mathbf{a}=\eta\left(\mathbf{v}^{(0)}-\mathbf{v}^{(k)}\right),
\quad
\Delta\mathbf{b}=\eta\left(\mathbf{p}^{+}-\mathbf{p}^{-}\right).
$$

Like the general BM CD-$k$ algorithm, this avoids waiting for full stationarity at every update; however, the RBM structure makes each Gibbs step much cheaper. In practice, CD-1 is often a strong baseline, while larger $k$ can reduce bias at increased computational cost.

___

## Lab
In the lab this week, we'll consider the question of how to train a Boltzmann Machine. In particular, we'll consider the problem of learning the weights $\mathbf{W}$ and biases $\mathbf{b}$ of a Boltzmann Machine from data. [Let's look at the training problem and solution algorithm developed by Hinton. et al called contrastive divergence](CHEME-5820-L7a-Training-BoltzmannMachines-Spring-2026.ipynb).

## Summary
This module introduced Boltzmann machines as stochastic recurrent neural networks that define probability distributions over binary state configurations.

> __Key Takeaways:__
>
> * **Boltzmann machine structure:** A Boltzmann machine is a fully connected undirected graph where each node has a binary state and bias, and edges have symmetric weights. The network defines a joint probability distribution over all possible state configurations.
> * **Gibbs sampling for state generation:** Because the partition function is intractable to compute directly, we use Gibbs sampling to generate states from the network. Each node updates stochastically based on the states of its neighbors and the logistic function of its total input.
> * **Convergence to stationary distribution:** After sufficient iterations, the sampled states converge to the Boltzmann distribution. Convergence can be monitored by tracking energy values across sliding windows.

The contrastive divergence algorithm enables training of Boltzmann machines by approximating likelihood gradients without computing the partition function.

___